In [1]:
import os

import ee

from utils import (
    get_export_periods,
    get_export_tile_ids,
    get_wgs84_region_bounds,
    get_region_pixel_size,
    get_region_crs,
    get_region_transform,
    gee_grid_tiles,
    get_long_region_name,
    get_long_ds_name,
    get_nodata_value
)

First, we must retrieve cloud bucket access credentials from the corresponding JSON file.

In [ ]:
# Replace with your equivalent path to credentials
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    "path/to/your/credentials.json"
)

In [3]:
ee.Authenticate()

True

In [ ]:
# Replace with your equivalent cloud project
ee.Initialize(project="your-cloud-project")

# Input arguments

In [5]:
# Bucket name
bucket = "isaac_shared"

# Region
short_region_name = "est"

# Dataset
short_ds_name = "lsat"

Uncomment all years in order to process the whole time series.

In [6]:
# List of years to process
years = [
    2017,
    2018,
    2019,
    2020,
    2021,
    # 2022,
    # 2023,
    # 2024,
    # 2025
]

In [7]:
# Dictionary of time periods to export
export_periods = get_export_periods(years, short_region_name)

In [8]:
print(export_periods)

{2017: {'start_dates': ['2017-04-01', '2017-06-01', '2017-09-01'], 'end_dates': ['2017-05-31', '2017-08-31', '2017-10-31']}, 2018: {'start_dates': ['2018-04-01', '2018-06-01', '2018-09-01'], 'end_dates': ['2018-05-31', '2018-08-31', '2018-10-31']}, 2019: {'start_dates': ['2019-04-01', '2019-06-01', '2019-09-01'], 'end_dates': ['2019-05-31', '2019-08-31', '2019-10-31']}, 2020: {'start_dates': ['2020-04-01', '2020-06-01', '2020-09-01'], 'end_dates': ['2020-05-31', '2020-08-31', '2020-10-31']}, 2021: {'start_dates': ['2021-04-01', '2021-06-01', '2021-09-01'], 'end_dates': ['2021-05-31', '2021-08-31', '2021-10-31']}}


You can uncomment the next cell if you wish to test the workflow with a single season instead of the full list.

In [9]:
export_periods = {}
for year in years:
    export_periods[year] = {
        "start_dates": [
            #f"{year}-04-01",
            f"{year}-06-01",
            #f"{year}-09-01"
        ],
        "end_dates": [
            #f"{year}-05-31",
            f"{year}-08-31",
            #f"{year}-10-31"
        ]
    }

Uncomment the variables you would like to process.

In [10]:
# List of variables to calculate
variables = [
    "ndvi",
    "evi",
    "savi",
    "fvc",
    "ndwi",
    "bsi",
    "ndmi",
    # "wiw",
    "lai",
    "gndvi",
    # "b1",
    # "b2",
    # "b3",
    # "b4",
    # "b5",
    # "b6",
    # "b7",
    # "b8",
]

In [11]:
# List of statistics to calculate
stats = [
    #"median",
    #"mean",
    "std",
    #"min",
    #"max"
]

In [12]:
# Dictionary mapping statistic keys to reducer functions
stat_reducers = {
    "median": lambda col: col.median(),
    "mean": lambda col: col.mean(),
    "std": lambda col: col.reduce(ee.Reducer.stdDev()),
    "min": lambda col: col.min(),
    "max": lambda col: col.max()
}

Tile IDs that are going to be exported are given in the following list. Only the tiles that intersect with the boundaries of the corresponding region (Estonia, Baltic states or European countries) will be exported.

In [13]:
# Tile IDs to export
export_tile_ids = get_export_tile_ids(short_region_name)

In [14]:
print(export_tile_ids)

['04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '18', '19']


You can uncomment the next cell if you wish to test the workflow with a single tile or a couple of tiles instead of the full list.

In [15]:
#export_tile_ids = ["13"]
#export_tile_ids = ['08', '09', '11', '12'] # this is probably index from old grid
#export_tile_ids = ['13', '17', '18'] # tile covering Estonia
#export_tile_ids = ['12']
#export_tile_ids = ['04', '05', '06', '07', '10', '11', '12', '13', '14', '15', '18', '19']
#print(export_tile_ids)

# Create output grid tiles

We will create an output grid where each tile is 10000$\times$10000 pixels, which is the default maximum output image size in GEE. This means that for 10 m resolution the `scale` parameter will be 100000 (100 km), for 30 m resolution 300000 (300 km) and for 100 m resolution 1000000 (1000 km).

In [16]:
# Region bounds in WGS84
region_bounds = get_wgs84_region_bounds(short_region_name)

# Pixel size
pixel_size = get_region_pixel_size(short_region_name)

# Region CRS
region_crs = get_region_crs(short_region_name)

# Calculate transform
region_transform = get_region_transform(short_region_name)

# Generate grid tiles
grid_tiles = gee_grid_tiles(region_bounds, region_crs, pixel_size)

In [17]:
print(region_crs)
print(region_transform)

EPSG:3301
[10, 0, 369000, 0, -10, 6635000]


# Calculate indices and export tiles

In [18]:
# Mask pixels based on probability of being clear
#def mask_low_qa(image, qa_band, clear_threshold=0.5):
#  mask = image.select(qa_band).gte(clear_threshold)
#  return image.updateMask(mask)

def maskL89sr(image):
    # QA_PIXEL bits
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow

    qa_mask = image.select('QA_PIXEL').bitwiseAnd(0b11111).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)
    mask = qa_mask.And(saturation_mask)
    
    # Apply scaling factors to optical and thermal bands
    #opticalBands = image.select(['SR_B.*']).multiply(0.0000275).add(-0.2)
    #thermalBands = image.select(['ST_B.*']).multiply(0.00341802).add(149.0)

    # Replace the original bands and apply masks
    #return (image.addBands(opticalBands, None, True)
                 #.addBands(thermalBands, None, True)
                 #.updateMask(qaMask)
                 #.updateMask(saturationMask)
                 #.copyProperties(image, image.propertyNames()))
    
    return image.updateMask(mask).copyProperties(image, image.propertyNames())

In [19]:
# Apply preprocessing steps to Landsat 8 collection
def preprocess_lsat_collection(start_date, end_date, grid_tiles, variable):

    # Define the required bands for each index
    required_bands = {
        "ndvi": ["SR_B4", "SR_B5"],
        "evi": ["SR_B2", "SR_B4", "SR_B5"],
        "savi": ["SR_B5", "SR_B4"],
        "fvc": ["SR_B4", "SR_B5"],
        "ndwi": ["SR_B3", "SR_B5"],
        "bsi": ["SR_B6", "SR_B4", "SR_B5", "SR_B2"],
        "ndmi": ["SR_B5", "SR_B6"],
        "wiw": ["SR_B5", "SR_B7"],
        "lai": ["SR_B2", "SR_B4", "SR_B5"],
        "gndvi": ["SR_B5", "SR_B3"],
        "b1": ["SR_B1"],
        "b2": ["SR_B2"],
        "b3": ["SR_B3"],
        "b4": ["SR_B4"],
        "b5": ["SR_B5"],
        "b6": ["SR_B6"],
        "b7": ["SR_B7"],
    }
    
    # Read Landsat 8 collection
    l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")

    # Add Landsat 9 collection
    l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")

    # Merge collection
    lsat = l8.merge(l9)
    
    # Filter Landsat 8 based on time and bounds
    lsat_filtered = lsat\
        .filterDate(
            ee.Date(start_date), ee.Date(end_date).advance(1, "day")
        )\
        .filterBounds(grid_tiles.geometry())
    
    # Set the desired bands based on the requested index
    desired_bands = required_bands.get(variable.lower(), [])
    #l8_filtered_reordered = l8_filtered\
    #    .map(lambda image: image.select(desired_bands))
    
    # Read Cloud Score+ collection
    #cs_plus = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
    #cs_plus_bands = ["cs", "cs_cdf"]
    
    # Add Cloud Score+ bands to Sentinel-2
    #s2_with_cs_plus = s2_filtered_reordered\
    #    .linkCollection(cs_plus, cs_plus_bands)

    # Mask clouds based on Cloud Score+
    #qa_band = "cs"
    #clear_threshold = 0.5
    #s2_masked = s2_with_cs_plus\
    #    .map(lambda img: mask_low_qa(img, qa_band, clear_threshold))
    
    lsat_masked = lsat_filtered.map(maskL89sr) #l8_filtered_reordered\
    
    return lsat_masked.map(lambda image: image.select(desired_bands))

In [20]:
# Calculate spectral index based on variable name for Sentinel-2 collection
def calc_lsat_index_band(lsat_collection, variable):
    
    # Normalize input to lowercase for consistency
    variable = variable.lower()

    # Skip index calculation for raw bands
    if variable in [
        "b1", "b2", "b3", "b4", 
        "b5", "b6", "b7"
    ]:
        index_band = (
            lsat_collection
            .select(variable.upper()).multiply(0.0000275).add(-0.2)
            .rename(variable)
        )
    
    # Normalized Difference Vegetation Index (NDVI)
    # https://custom-scripts.sentinel-hub.com/sentinel-2/ndvi/
    elif variable == "ndvi":

        scaled = lsat_collection.select(["SR_B5", "SR_B4"]).multiply(0.0000275).add(-0.2)
        
        # Formula: (NIR - RED) / (NIR + RED)
        index_band = scaled\
            .normalizedDifference(["SR_B5", "SR_B4"])\
            .rename(variable)
        
    # Enhanced Vegetation Index (EVI)
    elif variable.lower() == "evi":
        
        # Formula: 2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))
        evi = lsat_collection\
            .expression(
                "2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))",
                {
                    "NIR": lsat_collection.select("SR_B5").multiply(0.0000275).add(-0.2),
                    "RED": lsat_collection.select("SR_B4").multiply(0.0000275).add(-0.2),
                    "BLUE": lsat_collection.select("SR_B2").multiply(0.0000275).add(-0.2)
                }
            )

        # Constrain EVI to a realistic range
        lower_limit = -1
        upper_limit = 1
        index_band = (
            evi
            .where(evi.lt(lower_limit), lower_limit)
            .where(evi.gt(upper_limit), upper_limit)
            .rename(variable)
        )

    # Soil-Adjusted Vegetation Index (SAVI)
    # https://custom-scripts.sentinel-hub.com/sentinel-2/savi/
    elif variable == "savi":
        
        """
        Formula: ((NIR - RED) / (NIR + RED + L)) * (1 + L)
        
        Soil brightness correction factor (L) is defined as 0.5 to 
        accommodate most land cover types
        """
        savi = lsat_collection.expression(
            f"((NIR - RED) / (NIR + RED + 0.5)) * (1 + 0.5)",
            {
                "NIR": lsat_collection.select("SR_B5").multiply(0.0000275).add(-0.2),
                "RED": lsat_collection.select("SR_B4").multiply(0.0000275).add(-0.2)
            }
        )
        index_band = savi.rename(variable)

    # Fractional Vegetation Cover (FVC)
    # https://custom-scripts.sentinel-hub.com/sentinel-2/fcover/
    elif variable == "fvc":
        
        # Compute NDVI first
        ndvi = calc_lsat_index_band(lsat_collection, "ndvi")
        
        """
        Formula: (NDVI - NDVIs) / (NDVIv - NDVIs)
        
        Default values for NDVIs and NDVIv are taken from 
        https://custom-scripts.sentinel-hub.com/custom-scripts/landsat-8/land_surface_temperature_mapping/
        """
        ndvis = 0.2
        ndviv = 0.8
        fvc = ndvi.expression(
            "(NDVI - NDVIs) / (NDVIv - NDVIs)",
            {
                "NDVI": ndvi,
                "NDVIs": ndvis,
                "NDVIv": ndviv
            }
        )
        
        # Constrain FVC to a realistic range
        lower_limit = 0
        upper_limit = 1
        index_band = (
            fvc
            .where(fvc.lt(lower_limit), lower_limit)
            .where(fvc.gt(upper_limit), upper_limit)
            .rename(variable)
        )
        
    # Normalized Difference Water Index (NDWI)
    # https://custom-scripts.sentinel-hub.com/sentinel-2/ndwi/
    elif variable == "ndwi":

        scaled = lsat_collection.select(["SR_B3", "SR_B5"]).multiply(0.0000275).add(-0.2)
        
        # Formula: (GREEN - NIR) / (GREEN + NIR)
        index_band = scaled\
            .normalizedDifference(["SR_B3", "SR_B5"])\
            .rename(variable)

    # Bare Soil Index (BSI)
    # https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/barren_soil/
    elif variable == "bsi":
        
        # Formula: ((B11 + B4) - (B8 + B2)) / ((B11 + B4) + (B8 + B2))
        index_band = lsat_collection.expression(
            "((SR_B6 + SR_B4) - (SR_B5 + SR_B2)) / ((SR_B6 + SR_B4) + (SR_B5 + SR_B2))",
            {
                "SR_B6": lsat_collection.select("SR_B6").multiply(0.0000275).add(-0.2),
                "SR_B4": lsat_collection.select("SR_B4").multiply(0.0000275).add(-0.2),
                "SR_B5": lsat_collection.select("SR_B5").multiply(0.0000275).add(-0.2),
                "SR_B2": lsat_collection.select("SR_B2").multiply(0.0000275).add(-0.2)
            }
        ).rename(variable)
        
    # Normalized Difference Moisture Index (NDMI)
    # https://custom-scripts.sentinel-hub.com/sentinel-2/ndmi/
    elif variable == "ndmi":

        scaled = lsat_collection.select(["SR_B5", "SR_B6"]).multiply(0.0000275).add(-0.2)
        
        # Formula: (NIR - SWIR) / (NIR + SWIR)
        index_band = scaled\
            .normalizedDifference(["SR_B5", "SR_B6"])\
            .rename(variable)

    # Wetlands Index for Water (WIW)
    # https://custom-scripts.sentinel-hub.com/sentinel-2/wiw_s2_script/
    elif variable == "wiw":
        
        # Formula: 1 if (B8A <= 0.1804) and (B12 <= 0.1131), else 0
        nodata_mask = lsat_collection.select(["SR_B5", "SR_B7"]).mask()\
            .reduce(ee.Reducer.min())
        index_band = lsat_collection.expression(
            "(SR_B5 <= 0.1804) and (SR_B7 <= 0.1131) ? 1 : 0",
            {
                "SR_B5": lsat_collection.select("SR_B5").multiply(0.0000275).add(-0.2),
                "SR_B7": lsat_collection.select("SR_B7").multiply(0.0000275).add(-0.2)
            }
        ).rename(variable)
        
        # Apply the nodata mask and fill unmasked areas with nodata value
        nodata_val = 255
        index_band = index_band.updateMask(nodata_mask)\
            .unmask(value=nodata_val)

    # Leaf Area Index (LAI)
    # https://custom-scripts.sentinel-hub.com/sentinel-2/lai/
    elif variable == "lai":
        
        """
        Formula: LAI = (3.618 * EVI) - 0.118
        Based on the formula from Boegh et al. (2002), DOI: 
        https://doi.org/10.1016/S0034-4257(01)00342-X.
        Value range based on Boegh et al. (2002), DOI: 
        https://doi.org/10.1046/j.1466-822X.2003.00026.x.
        """
        evi = lsat_collection\
            .expression(
                "2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))",
                {
                    "NIR": lsat_collection.select("SR_B5").multiply(0.0000275).add(-0.2),
                    "RED": lsat_collection.select("SR_B4").multiply(0.0000275).add(-0.2),
                    "BLUE": lsat_collection.select("SR_B2").multiply(0.0000275).add(-0.2)
                }
            )
        lai = evi.expression(
            "(3.618 * EVI) - 0.118",
            {
                "EVI": evi
            }
        )
        
        # Constrain LAI to a realistic range
        lower_limit = 0
        upper_limit = 47
        index_band = (
            lai
            .where(lai.lt(lower_limit), lower_limit)
            .where(lai.gt(upper_limit), upper_limit)
            .rename(variable)
        )
        
    # Green Normalized Difference Vegetation Index (GNDVI)
    # https://custom-scripts.sentinel-hub.com/sentinel-2/gndvi/
    elif variable == "gndvi":

        scaled = lsat_collection.select(["SR_B5", "SR_B3"]).multiply(0.0000275).add(-0.2)
        
        # Formula: (NIR - GREEN) / (NIR + GREEN)
        index_band = scaled\
            .normalizedDifference(["SR_B5", "SR_B3"])\
            .rename(variable)
        
    return index_band

In [21]:
# List of grid IDs
grid_ids = grid_tiles.aggregate_array("system:index").getInfo()

In [22]:
# Load land mask from asset
if short_region_name == "est":
    land_mask_path = (
        "projects/ee-holgervirro/assets/"
        "est_ref_grid_500m_buffer_2025"
    )
    land_mask = ee.Image(land_mask_path)
elif short_region_name == "bal":
    land_mask_path = (
        "projects/ee-holgervirro/assets/"
        "bal_ref_grid_1km_buffer_2025"
    )
    land_mask = ee.Image(land_mask_path)
elif short_region_name == "eur":
    land_mask_path = (
        "projects/ee-holgervirro/assets/"
        "eur_ref_grid_1km_buffer_2025"
    )
    land_mask = ee.Image(land_mask_path)

In [23]:
# # Loop over years
# for year in export_periods.keys():
    
#     print(f"Year being processed: {year}")

#     # Start dates
#     start_dates = export_periods[year]["start_dates"]

#     # End dates
#     end_dates = export_periods[year]["end_dates"]

#     # Loop over variables
#     for variable in variables:
        
#         print(f"\tVariable being processed: {variable}")

#         # Nodata value
#         nodata_val = get_nodata_value(variable, short_region_name)

#         # Preprocess annual collection
#         lsat_preprocessed_annual = preprocess_lsat_collection(
#             f"{year}-01-01", f"{year}-12-31", grid_tiles, variable
#         )

#         # Resample annual collection based on region
#         if short_region_name in ["bal", "eur"]:
#             if variable != "wiw":
#                 lsat_preprocessed_annual = (
#                     lsat_preprocessed_annual
#                     .map(lambda img: img.resample("bilinear"))
#                 )

#         # Calculate index per image in annual collection
#         index_collection_annual = (
#             lsat_preprocessed_annual
#             .map(lambda img: calc_lsat_index_band(img, variable))
#             .select(variable)
#         )

#         # Loop over statistics
#         for stat in stats:

#             print(f"\t\tStatistic being processed: {stat}")
            
#             # Get reducer function
#             reducer_fn = stat_reducers[stat]

#             # Calculate annual statistic
#             index_annual_stat = (
#                 reducer_fn(index_collection_annual)
#                 .rename(f"{variable}_{stat}")
#                 .unmask(value=nodata_val)
#             )

#             # Loop over time periods
#             for start_date, end_date in zip(start_dates, end_dates):
                
#                 print(f"\t\t\tTime period start date: {start_date}")
#                 print(f"\t\t\tTime period end date: {end_date}")

#                 # Preprocess seasonal collection
#                 lsat_preprocessed = preprocess_lsat_collection(
#                     start_date, end_date, grid_tiles, variable
#                 )

#                 # Resample seasonal collection based on region
#                 if short_region_name in ["bal", "eur"]:
#                     if variable != "wiw":
#                         lsat_preprocessed = (
#                             lsat_preprocessed
#                             .map(lambda img: img.resample("bilinear"))
#                         )

#                 # Calculate index per image in seasonal collection
#                 index_collection_seasonal = (
#                     lsat_preprocessed
#                     .map(lambda img: calc_lsat_index_band(img, variable))
#                     .select(variable)
#                 )

#                 # Calculate seasonal statistic
#                 index_season_stat = (
#                     reducer_fn(index_collection_seasonal)
#                     .rename(f"{variable}_{stat}")
#                 )

#                 # Create a mask for nodata pixels
#                 nodata_mask = (
#                     index_season_stat
#                     .unmask(value=nodata_val)
#                     .eq(nodata_val)
#                 )
                
#                 # Fill masked pixels with annual values
#                 image_filled = (
#                     index_season_stat
#                     .unmask(index_annual_stat)
#                     .where(
#                         index_season_stat.eq(nodata_val),
#                         index_annual_stat
#                     )
#                 )

#                 # Replace masked pixels with the nodata value
#                 unmasked_image = image_filled.unmask(value=nodata_val)

#                 # Skip scaling for raw bands
#                 if variable in [
#                     "b1", "b2", "b3", "b4", "b5", "b6",
#                     "b7", "b8",
#                 ]:
#                     out_image = unmasked_image.toInt16()
                
#                 # Skip scaling for binary WIW image
#                 elif variable == "wiw":
#                     out_image = unmasked_image
                
#                 else:
                    
#                     # Scale values
#                     out_image = (
#                         unmasked_image.where(nodata_mask, 0)
#                         .multiply(1000)
#                         .round()
#                     )
#                     if variable == "lai":
#                         out_image = out_image.toInt32()
#                     else:
#                         out_image = out_image.toInt16()
                
#                 # Apply land mask
#                 #if short_region_name in ["bal", "eur"]:
#                 if short_region_name in ["est", "bal", "eur"]:
#                     out_image = (
#                         out_image.where(nodata_mask, nodata_val)
#                         #.updateMask(land_mask)
#                         .unmask(nodata_val)
#                     )
#                 else:
#                     out_image = out_image.where(nodata_mask, nodata_val)
                    
#                 # Loop over grid tile IDs
#                 for i, grid_id in enumerate(grid_ids):

#                     # Output tile ID
#                     tile_id = str(i + 1).zfill(2)
                    
#                     if tile_id in export_tile_ids:

#                         # Get tile geometry
#                         feature = ee.Feature(grid_tiles.toList(1, i).get(0))
#                         region = feature.geometry()
                        
#                         # Launch export task for filled image tile
#                         filename = "_".join(
#                             [
#                                 short_region_name,
#                                 short_ds_name,
#                                 f"{variable}_{stat}",
#                                 start_date,
#                                 end_date,
#                                 tile_id
#                             ]
#                         )
#                         task_name = f"export_{filename}"
#                         long_region_name = get_long_region_name(
#                             short_region_name
#                         )
#                         long_ds_name = get_long_ds_name(
#                             short_ds_name
#                         )
#                         prefix = (
#                             #f"{long_region_name}/"
#                             f"{long_region_name}_working_rev/"
#                             f"{long_ds_name}/"
#                             f"{variable}/"
#                             f"{year}/"
#                             #f"{filename}"
#                             f"{filename}_30m_l7"
#                         )
#                         task = ee.batch.Export.image.toCloudStorage(
#                             image=out_image,
#                             description=task_name,
#                             bucket=bucket,
#                             fileNamePrefix=prefix,
#                             fileFormat="GeoTIFF",
#                             crs=region_crs,
#                             crsTransform=region_transform,
#                             scale=30,
#                             maxPixels=1e13,
#                             region=region,
#                             formatOptions={
#                                 "cloudOptimized": True,
#                                 "noData": nodata_val
#                             }
#                         )
#                         task.start()
#                         print(f"\t\t\t\tStarted task: {task_name}")
                        
#                         # Launch export task for nodata tile
#                         #nodata_filename = "_".join(
#                         #     [
#                         #         short_region_name,
#                         #         short_ds_name,
#                         #         f"{variable}",
#                         #         start_date,
#                         #         end_date,
#                         #         tile_id,
#                         #         "nodata"
#                         #     ]
#                         # )
#                         #nodata_task_name = f"export_{nodata_filename}"
#                         #nodata_prefix = (
#                         #    f"{long_region_name}_nodata/"
#                         #    f"{long_ds_name}/"
#                         #    f"{variable}/"
#                         #    f"{year}/"
#                         #    f"{nodata_filename}"
#                         #)
#                         #nodata_task = ee.batch.Export.image.toCloudStorage(
#                         #    image=nodata_mask,
#                         #    description=nodata_task_name,
#                         #    bucket=bucket,
#                         #    fileNamePrefix=nodata_prefix,
#                         #    fileFormat="GeoTIFF",
#                         #    crs=region_crs,
#                         #    crsTransform=region_transform,
#                         #    maxPixels=1e13,
#                         #    region=region,
#                         #    formatOptions={
#                         #        "cloudOptimized": True,
#                         #        "noData": 0
#                         #    }
#                         #)
#                         #nodata_task.start()
#                         #print(f"\t\t\t\tStarted task: {nodata_task_name}")
                    
#     print("\n")

In [24]:
# Loop over years
for year in export_periods.keys():
    
    print(f"Year being processed: {year}")

    start_dates = export_periods[year]["start_dates"]
    end_dates = export_periods[year]["end_dates"]

    for variable in variables:
        
        print(f"\tVariable being processed: {variable}")

        nodata_val = get_nodata_value(variable, short_region_name)

        # --- Annual collection ---
        lsat_preprocessed_annual = preprocess_lsat_collection(
            f"{year}-01-01", f"{year}-12-31", grid_tiles, variable
        )

        if short_region_name in ["bal", "eur"] and variable != "wiw":
            lsat_preprocessed_annual = (
                lsat_preprocessed_annual
                .map(lambda img: img.resample("bilinear"))
            )

        index_collection_annual = (
            lsat_preprocessed_annual
            .map(lambda img: calc_lsat_index_band(img, variable))
            .select(variable)
        )

        for stat in stats:

            print(f"\t\tStatistic being processed: {stat}")
            
            reducer_fn = stat_reducers[stat]

            index_annual_stat = (
                reducer_fn(index_collection_annual)
                .rename(f"{variable}_{stat}")
            )
            # 🔧 REMOVED: .unmask(value=nodata_val)

            for start_date, end_date in zip(start_dates, end_dates):
                
                print(f"\t\t\tTime period start date: {start_date}")
                print(f"\t\t\tTime period end date: {end_date}")

                # --- Seasonal collection ---
                lsat_preprocessed = preprocess_lsat_collection(
                    start_date, end_date, grid_tiles, variable
                )

                if short_region_name in ["bal", "eur"] and variable != "wiw":
                    lsat_preprocessed = (
                        lsat_preprocessed
                        .map(lambda img: img.resample("bilinear"))
                    )

                index_collection_seasonal = (
                    lsat_preprocessed
                    .map(lambda img: calc_lsat_index_band(img, variable))
                    .select(variable)
                )

                index_season_stat = (
                    reducer_fn(index_collection_seasonal)
                    .rename(f"{variable}_{stat}")
                )

                # ============================================================
                # 🔧🔧🔧 CORE FIX: fill seasonal nodata with annual values
                # ============================================================

                # Seasonal mask (true = valid data)
                season_mask = index_season_stat.mask()   # 🔧 NEW

                # Fill only masked seasonal pixels using annual stat
                filled_image = index_season_stat.unmask(index_annual_stat)  # 🔧 NEW

                # Pixels are valid if either seasonal or annual is valid
                combined_mask = (
                    season_mask
                    .Or(index_annual_stat.mask())
                )  # 🔧 NEW

                filled_image = filled_image.updateMask(combined_mask)  # 🔧 NEW

                # ============================================================

                # --- Scaling / datatype logic ---
                if variable in [
                    "b1", "b2", "b3", "b4", "b5", "b6",
                    "b7", "b8",
                ]:
                    out_image = filled_image.toInt16()

                elif variable == "wiw":
                    out_image = filled_image

                else:
                    out_image = (
                        filled_image
                        .multiply(1000)
                        .round()
                    )

                    if variable == "lai":
                        out_image = out_image.toInt32()
                    else:
                        out_image = out_image.toInt16()

                # ============================================================
                # 🔧 Apply nodata value ONLY ONCE, AT THE END
                # ============================================================
                out_image = out_image.unmask(nodata_val)   # 🔧 NEW

                # --- Export ---
                for i, grid_id in enumerate(grid_ids):

                    tile_id = str(i + 1).zfill(2)
                    
                    if tile_id in export_tile_ids:

                        feature = ee.Feature(grid_tiles.toList(1, i).get(0))
                        region = feature.geometry()
                        
                        filename = "_".join(
                            [
                                short_region_name,
                                short_ds_name,
                                f"{variable}_{stat}",
                                start_date,
                                end_date,
                                tile_id
                            ]
                        )

                        task_name = f"export_{filename}"

                        long_region_name = get_long_region_name(short_region_name)
                        long_ds_name = get_long_ds_name(short_ds_name)

                        prefix = (
                            f"{long_region_name}_working"
                            f"{long_ds_name}/"
                            f"{variable}/"
                            f"{year}/"
                            f"{filename}_30m"
                        )

                        task = ee.batch.Export.image.toCloudStorage(
                            image=out_image,
                            description=task_name,
                            bucket=bucket,
                            fileNamePrefix=prefix,
                            fileFormat="GeoTIFF",
                            crs=region_crs,
                            crsTransform=region_transform,
                            scale=30,
                            maxPixels=1e13,
                            region=region,
                            formatOptions={
                                "cloudOptimized": True,
                                "noData": nodata_val
                            }
                        )

                        task.start()
                        print(f"\t\t\t\tStarted task: {task_name}")

    print("\n")


Year being processed: 2017
	Variable being processed: ndvi
		Statistic being processed: std
			Time period start date: 2017-06-01
			Time period end date: 2017-08-31
				Started task: export_est_lsat_ndvi_std_2017-06-01_2017-08-31_04
				Started task: export_est_lsat_ndvi_std_2017-06-01_2017-08-31_05
				Started task: export_est_lsat_ndvi_std_2017-06-01_2017-08-31_06
				Started task: export_est_lsat_ndvi_std_2017-06-01_2017-08-31_07
				Started task: export_est_lsat_ndvi_std_2017-06-01_2017-08-31_08
				Started task: export_est_lsat_ndvi_std_2017-06-01_2017-08-31_09
				Started task: export_est_lsat_ndvi_std_2017-06-01_2017-08-31_10
				Started task: export_est_lsat_ndvi_std_2017-06-01_2017-08-31_11
				Started task: export_est_lsat_ndvi_std_2017-06-01_2017-08-31_12
				Started task: export_est_lsat_ndvi_std_2017-06-01_2017-08-31_13
				Started task: export_est_lsat_ndvi_std_2017-06-01_2017-08-31_14
				Started task: export_est_lsat_ndvi_std_2017-06-01_2017-08-31_15
				Started task: 